In [ ]:
# ============================================================
# 5. EKF Main Loop (True Plant 분리 및 RMSE 측정을 위한 정석 루프)
# ============================================================
n_steps = 4000
x_true = x0.copy()  # 실제 차량의 물리적 상태 (True Plant)
x_est = x0.copy()   # EKF가 추정한 상태 (Estimator)
P_est = P0.copy()

true_history = []
est_history = []

print("Running EKF with True Plant & Augmented State...")
start_exec = time.time()

for k in range(n_steps):
    # --- 1. True Plant Propagation (실제 차량의 움직임) ---
    # 실제 차량은 프로세스 노이즈를 겪으며 이동함
    w_true = np.random.multivariate_normal(np.zeros(5), Q).reshape(-1, 1)
    w_true[4, 0] = 0  # 바이어스는 상수이므로 노이즈 없이 고정
    
    x_true = f_discrete(x_true, h) + w_true
    x_true[4, 0] = TRUE_BIAS  # 우측 상단 비콘에 물리적으로 1.5m 에러 강제 주입
    
    # 지침서 제약 조건: 실제 차량은 차선(2m)을 절대 넘지 않음 (Soft boundary 방어)
    if x_true[2, 0] > 2.0: x_true[2, 0] = 2.0
    elif x_true[2, 0] < -2.0: x_true[2, 0] = -2.0

    # --- 2. Generate Real Measurement (실제 센서 측정값 생성) ---
    z_real = h_measurement(x_true) + np.random.normal(0, R_meas_std, (4, 1))

    # --- 3. EKF Prediction (시간 업데이트) ---
    x_pred = f_discrete(x_est, h)
    A_k = get_A_matrix(x_est, h)
    P_pred = A_k @ P_est @ A_k.T + Q
    P_pred = (P_pred + P_pred.T) / 2.0

    # --- 4. EKF Correction (측정 업데이트) ---
    C_k = get_C_matrix(x_pred)
    z_hat = h_measurement(x_pred)
    
    innovation = z_real - z_hat
    S = C_k @ P_pred @ C_k.T + R_mat
    K = P_pred @ C_k.T @ np.linalg.inv(S)
    
    x_est = x_pred + K @ innovation
    P_est = (np.eye(5) - K @ C_k) @ P_pred
    P_est = (P_est + P_est.T) / 2.0

    true_history.append(x_true.flatten())
    est_history.append(x_est.flatten())

exec_time = (time.time() - start_exec) / n_steps

# ============================================================
# 6. Evaluation (RMSE 계산)
# ============================================================
true_history = np.array(true_history)
est_history = np.array(est_history)
t_axis = np.arange(n_steps) * h

# RMSE 계산 (Burn-in 기간 초기 100스텝 제외하고 계산하면 더 정확한 정상상태 성능 측정 가능)
rmse_s = np.sqrt(np.mean((true_history[100:, 0] - est_history[100:, 0])**2))
rmse_d = np.sqrt(np.mean((true_history[100:, 2] - est_history[100:, 2])**2))
bias_error = abs(TRUE_BIAS - est_history[-1, 4])

print("\n--- Step 4 Performance Metrics ---")
print(f"Avg Execution Time : {exec_time:.6f}s per step (Real-time feasibility: OK)")
print(f"Longitudinal RMSE  : {rmse_s:.4f} m")
print(f"Lateral (d) RMSE   : {rmse_d:.4f} m")
print(f"Final Bias Estimate: {est_history[-1, 4]:.4f} m (Error: {bias_error:.4f} m)")

# ============================================================
# 7. Result Visualization (보고서용 3종 그래프)
# ============================================================
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

# [Graph 1] Lateral Offset Tracking (차선 유지 추적 성능)
ax1.plot(t_axis, true_history[:, 2], 'b-', label='True Lateral Offset (Plant)', alpha=0.8)
ax1.plot(t_axis, est_history[:, 2], 'r--', label='Estimated Offset (EKF)')
ax1.axhline(2.0, color='k', linestyle=':', label='Lane Boundary (+2m)')
ax1.axhline(-2.0, color='k', linestyle=':')
ax1.set_title(f'Lateral Tracking Performance (RMSE: {rmse_d:.4f} m)')
ax1.set_ylabel('Lateral Offset [m]')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper right')

# [Graph 2] Sensor Bias Convergence (바이어스 수렴)
ax2.plot(t_axis, est_history[:, 4], 'g-', linewidth=2, label='Estimated Bias (State 5)')
ax2.axhline(TRUE_BIAS, color='r', linestyle='--', label='True Injected Bias (1.5m)')
ax2.set_title('Sensor Bias Convergence over Time')
ax2.set_xlabel('Time [s]')
ax2.set_ylabel('Bias [m]')
ax2.grid(True, alpha=0.3)
ax2.legend(loc='lower right')

plt.tight_layout()
plt.show()

# [Graph 3] 2D Trajectory Map (글로벌 궤적 및 비콘 배치)
# Global X, Y 좌표를 계산하여 그립니다.
X_true, Y_true = get_global_pos(true_history[:, 0], true_history[:, 2])
X_est, Y_est = get_global_pos(est_history[:, 0], est_history[:, 2])

plt.figure(figsize=(10, 6))
plt.plot(X_true, Y_true, 'b-', label='True Trajectory', linewidth=2)
plt.plot(X_est, Y_est, 'r--', label='Estimated Trajectory')
plt.scatter(beacons[:, 0], beacons[:, 1], c='black', marker='^', s=100, label='Beacons (B3 has bias)')

# 비콘 라벨링
for i, (bx, by) in enumerate(beacons):
    plt.text(bx + 2, by + 2, f'B{i+1}', fontsize=12, fontweight='bold')

plt.title('2D Track Trajectory and Rectangular Beacon Layout')
plt.xlabel('Global X [m]')
plt.ylabel('Global Y [m]')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()